In [13]:
# Cell 1: Imports and load raw data
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

hiaval = pd.read_csv('HiAVALDB.csv', na_values=['NA', 'na', 'N/A', ''],
                     encoding='latin-1')
era5   = pd.read_csv('era5_uk_hp_nepal.csv', encoding='latin-1')

print(f"HiAVALDB raw: {hiaval.shape}")
print(f"ERA5 raw:     {era5.shape}")
hiaval.head()

HiAVALDB raw: (746, 16)
ERA5 raw:     (688, 11)


,Location,Year,Month,Day,Latitude,Longitude,Country,Region_HiMAP,Type,Impact,Fatalities,Injured,Livestock,Leisure,Remarks,Reference
0,Talgar Pass,1951,11.0,NaN,43.114748,77.111547,Kazakhstan,4.0,NaN,Y,5.0,3.0,NaN,Y,NaN,https://edoc.ub.uni-muenchen.de/7262/1/Yegorov...
1,Tuyuksu Gletscher,1958,11.0,NaN,43.040461,77.039701,Kazakhstan,4.0,NaN,Y,1.0,NaN,NaN,NaN,NaN,https://edoc.ub.uni-muenchen.de/7262/1/Yegorov...
2,Pioneer Pass,1961,3.0,NaN,42.545453,74.548544,Kazakhstan,4.0,NaN,Y,1.0,1.0,NaN,Y,NaN,https://edoc.ub.uni-muenchen.de/7262/1/Yegorov...
3,Komsomol Peak,1966,3.0,NaN,42.571172,74.546949,Kazakhstan,4.0,NaN,Y,2.0,2.0,NaN,NaN,NaN,https://edoc.ub.uni-muenchen.de/7262/1/Yegorov...
4,Deo-Tibba,1972,10.0,28.0,32.186436,77.383044,India,10.0,NaN,Y,2.0,NaN,NaN,NaN,heavy snowfall triggered,"(Chandel et al., 2015)"


In [14]:
# Cell 2: Convert numeric columns in HiAVALDB
hiaval['Year']      = pd.to_numeric(hiaval['Year'], errors='coerce')
hiaval['Month']     = pd.to_numeric(hiaval['Month'], errors='coerce')
hiaval['Day']       = pd.to_numeric(hiaval['Day'], errors='coerce')
hiaval['Latitude']  = pd.to_numeric(hiaval['Latitude'], errors='coerce')
hiaval['Longitude'] = pd.to_numeric(hiaval['Longitude'], errors='coerce')

print("Data types after conversion:")
print(hiaval[['Year','Month','Day','Latitude','Longitude']].dtypes)
print(f"\nMissing values:\n{hiaval[['Year','Month','Day','Latitude','Longitude']].isnull().sum()}")

Data types after conversion:
Year         float64
Month        float64
Day          float64
Latitude     float64
Longitude    float64
dtype: object

Missing values:
Year          1
Month        13
Day          49
Latitude     31
Longitude    30
dtype: int64


In [15]:
# Cell 3: Fallback for missing month/day + drop rows missing Year/Lat/Lon
hiaval['Month'] = hiaval['Month'].fillna(7)
hiaval['Day']   = hiaval['Day'].fillna(15)

hiaval_clean = hiaval.dropna(subset=['Year', 'Latitude', 'Longitude']).copy()

print(f"Rows before drop: {len(hiaval)}")
print(f"Rows after drop:  {len(hiaval_clean)}")

Rows before drop: 746
Rows after drop:  715


In [16]:
# Cell 4: Winter filter + exclude glacier detachments
winter_months = [11, 12, 1, 2, 3, 4]

hiaval_winter = hiaval_clean[hiaval_clean['Month'].isin(winter_months)].copy()

hiaval_winter = hiaval_winter[
    ~hiaval_winter['Type'].astype(str).str.contains('glacier detachment', case=False, na=False)
]

print(f"Winter avalanche events (after excluding glacier detachments): {len(hiaval_winter)}")

Winter avalanche events (after excluding glacier detachments): 657


In [17]:
# Cell 5: Build date column and round coordinates for joining
hiaval_winter['date'] = pd.to_datetime(
    dict(year=hiaval_winter['Year'].astype(int),
         month=hiaval_winter['Month'].astype(int),
         day=hiaval_winter['Day'].astype(int)),
    errors='coerce'
)

hiaval_winter = hiaval_winter.dropna(subset=['date'])

hiaval_winter['lat_r'] = hiaval_winter['Latitude'].round(6)
hiaval_winter['lon_r'] = hiaval_winter['Longitude'].round(6)

print(f"Final winter events with valid dates: {len(hiaval_winter)}")
print(f"Date range: {hiaval_winter['date'].min()} to {hiaval_winter['date'].max()}")

Final winter events with valid dates: 656
Date range: 1951-11-15 00:00:00 to 2025-12-18 00:00:00


In [18]:
# Cell 6: Clean and filter ERA5
era5['date']  = pd.to_datetime(era5['date_str'])
era5['lat_r'] = era5['latitude'].round(6)
era5['lon_r'] = era5['longitude'].round(6)

# Deduplicate on date + location
era5 = era5.drop_duplicates(subset=['date', 'lat_r', 'lon_r'])

# Filter to winter months
era5['month'] = era5['date'].dt.month
era5_winter = era5[era5['month'].isin(winter_months)].copy()

print(f"ERA5 total rows:  {len(era5)}")
print(f"ERA5 winter rows: {len(era5_winter)}")
print(f"ERA5 date range:  {era5['date'].min()} to {era5['date'].max()}")

ERA5 total rows:  687
ERA5 winter rows: 223
ERA5 date range:  1950-01-02 00:00:00 to 2024-11-11 00:00:00


In [19]:
# Cell 7: Join winter avalanche events to ERA5 (positives)
positives = hiaval_winter.merge(
    era5_winter,
    on=['date', 'lat_r', 'lon_r'],
    how='inner',
    suffixes=('', '_era5')
)
positives['target'] = 1

# Deduplicate positives
positives = positives.drop_duplicates(subset=['date', 'lat_r', 'lon_r'])

print(f"Positives matched to ERA5: {len(positives)} / {len(hiaval_winter)}")
if len(hiaval_winter) > 0:
    print(f"Match rate: {len(positives)/len(hiaval_winter)*100:.1f}%")

Positives matched to ERA5: 31 / 656
Match rate: 4.7%


In [20]:
# Cell 8: Build negatives from ERA5 rows with no avalanche
era5_winter['key'] = (era5_winter['date'].astype(str) + '_' +
                      era5_winter['lat_r'].astype(str) + '_' +
                      era5_winter['lon_r'].astype(str))

positives['key'] = (positives['date'].astype(str) + '_' +
                    positives['lat_r'].astype(str) + '_' +
                    positives['lon_r'].astype(str))

negatives = era5_winter[~era5_winter['key'].isin(positives['key'])].copy()
negatives['target'] = 0

print(f"Negatives: {len(negatives)}")
print(f"Total samples: {len(positives) + len(negatives)}")
print(f"Class balance: {len(positives)} pos / {len(negatives)} neg")

Negatives: 192
Total samples: 223
Class balance: 31 pos / 192 neg


In [21]:
# Cell 9: Feature engineering function
def engineer_features(df):
    """Convert ERA5 units and engineer derived features."""
    df = df.copy()

    # Temperature: Kelvin -> Celsius
    df['temperature_C'] = df['temperature_2m'] - 273.15
    df['dewpoint_C']    = df['dewpoint_temperature_2m'] - 273.15

    # Precipitation: meters -> mm
    df['precip_mm']     = df['total_precipitation_sum'] * 1000
    df['snowfall_mm']   = df['snowfall_sum'] * 1000

    # Snow depth: meters -> mm, clip at ERA5 cap (10 m = 10000 mm)
    df['snow_depth_mm'] = df['snow_depth_water_equivalent'] * 1000
    df['snow_depth_mm'] = df['snow_depth_mm'].clip(lower=0, upper=9999)

    # Pressure: Pa -> hPa
    df['pressure_hPa'] = df['surface_pressure'] / 100

    # Wind speed from u,v components
    df['wind_speed'] = np.sqrt(df['u_component_of_wind_10m']**2 +
                               df['v_component_of_wind_10m']**2)

    # Relative humidity via Magnus formula
    T  = df['temperature_C']
    Td = df['dewpoint_C']
    df['relative_humidity'] = 100 * (np.exp((17.625 * Td) / (243.04 + Td)) /
                                     np.exp((17.625 * T)  / (243.04 + T)))
    df['relative_humidity'] = df['relative_humidity'].clip(0, 100)

    # Month (seasonality)
    if 'date' in df.columns:
        df['month'] = pd.to_datetime(df['date']).dt.month

    return df

print("engineer_features() defined.")

engineer_features() defined.


In [22]:
# Cell 10: Apply feature engineering and define feature list
positives = engineer_features(positives)
negatives = engineer_features(negatives)

# Expanded feature set: weather + location + seasonality
feature_cols = [
    'temperature_C', 'dewpoint_C', 'precip_mm', 'snowfall_mm',
    'snow_depth_mm', 'pressure_hPa', 'wind_speed', 'relative_humidity',
    'latitude', 'longitude', 'month'
]

print("Features engineered:")
for col in feature_cols:
    print(f"  {col}")

print(f"\nPositives preview:")
print(positives[feature_cols].describe().T)

Features engineered:
  temperature_C
  dewpoint_C
  precip_mm
  snowfall_mm
  snow_depth_mm
  pressure_hPa
  wind_speed
  relative_humidity
  latitude
  longitude
  month

Positives preview:
                   count        mean          std         min         25%  \
temperature_C       31.0   -7.374404     5.724294  -26.236498  -10.381982   
dewpoint_C          31.0  -11.969097     7.294504  -34.018570  -14.675792   
precip_mm           31.0    8.251987    14.559763    0.000435    0.027829   
snowfall_mm         31.0    6.703859    12.194029    0.000000    0.012545   
snow_depth_mm       31.0  931.724797  1720.041020    0.347853  109.929840   
pressure_hPa        31.0  646.848184    80.349635  552.190693  583.027418   
wind_speed          31.0    0.676653     0.381058    0.065390    0.461967   
relative_humidity   31.0   71.569446    17.501886   27.664885   56.647921   
latitude            31.0   30.733173     2.512834   27.376775   28.487476   
longitude           31.0   80.394698   

In [23]:
# Cell 11: Combine positives + negatives and save final CSV
pos_out = positives[['date', 'lat_r', 'lon_r', 'target'] + feature_cols].copy()
pos_out = pos_out.rename(columns={'lat_r': 'latitude', 'lon_r': 'longitude'})

neg_out = negatives[['date', 'lat_r', 'lon_r', 'target'] + feature_cols].copy()
neg_out = neg_out.rename(columns={'lat_r': 'latitude', 'lon_r': 'longitude'})

# Keep only one copy of lat/lon (feature_cols already has 'latitude','longitude')
pos_out = pos_out.loc[:, ~pos_out.columns.duplicated()]
neg_out = neg_out.loc[:, ~neg_out.columns.duplicated()]

final = pd.concat([pos_out, neg_out], ignore_index=True)
final = final.sort_values('date').reset_index(drop=True)

final.to_csv('avalanche_training_final.csv', index=False)

print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"Final training set: {len(final)} rows")
print(f"  - Positives: {final['target'].sum()}")
print(f"  - Negatives: {(final['target'] == 0).sum()}")
print(f"  - Date range: {final['date'].min()} to {final['date'].max()}")
print(f"  - Features: {feature_cols}")
print(f"\nSaved to avalanche_training_final.csv")
print(f"\nFinal columns: {list(final.columns)}")

PREPROCESSING COMPLETE
Final training set: 223 rows
  - Positives: 31
  - Negatives: 192
  - Date range: 1950-01-02 00:00:00 to 2024-11-11 00:00:00
  - Features: ['temperature_C', 'dewpoint_C', 'precip_mm', 'snowfall_mm', 'snow_depth_mm', 'pressure_hPa', 'wind_speed', 'relative_humidity', 'latitude', 'longitude', 'month']

Saved to avalanche_training_final.csv

Final columns: ['date', 'latitude', 'longitude', 'target', 'temperature_C', 'dewpoint_C', 'precip_mm', 'snowfall_mm', 'snow_depth_mm', 'pressure_hPa', 'wind_speed', 'relative_humidity', 'month']


In [24]:
# Cell 12: Sanity check the final dataset
print("Target distribution:")
print(final['target'].value_counts())

print("\nMissing values per feature:")
print(final[feature_cols].isnull().sum())

print("\nFeature correlations with target:")
corr = final[feature_cols + ['target']].corr()['target'].sort_values(ascending=False)
print(corr)

print("\nFeature means by class:")
for col in feature_cols:
    pos_mean = final[final['target'] == 1][col].mean()
    neg_mean = final[final['target'] == 0][col].mean()
    print(f"  {col:20s}: pos={pos_mean:10.3f}, neg={neg_mean:10.3f}")

print(f"\nSnow depth cap check (rows at 9999): {(final['snow_depth_mm'] >= 9999).sum()}")

Target distribution:
target
0    192
1     31
Name: count, dtype: int64

Missing values per feature:
temperature_C        0
dewpoint_C           0
precip_mm            0
snowfall_mm          0
snow_depth_mm        0
pressure_hPa         0
wind_speed           0
relative_humidity    0
latitude             0
longitude            0
month                0
dtype: int64

Feature correlations with target:
target               1.000000
relative_humidity    0.312140
precip_mm            0.280226
snowfall_mm          0.263105
snow_depth_mm        0.097130
latitude             0.052518
longitude           -0.021890
dewpoint_C          -0.047785
month               -0.105317
pressure_hPa        -0.155263
temperature_C       -0.172619
wind_speed          -0.227990
Name: target, dtype: float64

Feature means by class:
  temperature_C       : pos=    -7.374, neg=    -0.299
  dewpoint_C          : pos=   -11.969, neg=   -10.000
  precip_mm           : pos=     8.252, neg=     1.919
  snowfall_mm      